In [115]:
import pandas as pd

In [116]:
sensor_columns = [f'sensor{i}' for i in range(1, 22)]

In [117]:
cols = ['engine_id', 'cycle', 'op1', 'op2', 'op3'] + sensor_columns

In [118]:
train_df = pd.read_csv('../data/raw/train_FD001.txt', sep=r'\s+', header=None, names=cols)
test_df = pd.read_csv('../data/raw/test_FD001.txt', sep=r'\s+', header=None, names=cols)

In [119]:
train_df.head()

,engine_id,cycle,op1,op2,op3,sensor1,sensor2,sensor3,sensor4,sensor5,...,sensor12,sensor13,sensor14,sensor15,sensor16,sensor17,sensor18,sensor19,sensor20,sensor21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [120]:
train_df.columns

Index(['engine_id', 'cycle', 'op1', 'op2', 'op3', 'sensor1', 'sensor2',
       'sensor3', 'sensor4', 'sensor5', 'sensor6', 'sensor7', 'sensor8',
       'sensor9', 'sensor10', 'sensor11', 'sensor12', 'sensor13', 'sensor14',
       'sensor15', 'sensor16', 'sensor17', 'sensor18', 'sensor19', 'sensor20',
       'sensor21'],
      dtype='object')

In [121]:
# Step 1 - add RUL column to train_df
# Calculate how many flights each engine has left at every point in its history. You do this by
# finding the last cycle each engine reached, then subtracting the current cycle from that number.
# This becomes the target your model learns to predict.

engine_ids = train_df['engine_id'].unique().tolist()

max_cycle = train_df.groupby('engine_id')['cycle'].transform('max')
current_cycle = train_df['cycle']

train_df['RUL'] = max_cycle - current_cycle

In [122]:
# Step 2 - drop low-variance sensors from both train_df and test_df
# Identify which of the 21 sensors barely change across an engine's lifetime --
# those carry no useful informaton about degradation. Remove them from both datafames. 
# You calculate which ones to drop using only the training data, then apply that same list to the test data

column_variances = train_df[sensor_columns].var().sort_values(ascending=True)

low_variance_columns = column_variances[column_variances < 0.01].index

train_df = train_df.drop(columns=low_variance_columns)

In [123]:
test_df = test_df.drop(columns=low_variance_columns)

In [156]:
# Adding running averages and standard deviations for each sensor to the dataframe over the last 5 flights and over the last 10 flights to each engine (1 - 100)

In [ ]:
sensor_cols = [column for column in train_df.columns.tolist() if column.startswith('sensor')]

In [142]:
for column in sensor_cols:
    for w in [5, 10]:
        train_df[f'{column}_mean{w}'] = train_df.groupby('engine_id')[column].rolling(window=w, min_periods=1).mean().reset_index(level=0, drop=True)
        train_df[f'{column}_std{w}'] = train_df.groupby('engine_id')[column].rolling(window=w, min_periods=1).std().fillna(0).reset_index(level=0, drop=True)
        
        test_df[f'{column}_mean{w}'] = test_df.groupby('engine_id')[column].rolling(window=w, min_periods=1).mean().reset_index(level=0, drop=True)
        test_df[f'{column}_std{w}'] = test_df.groupby('engine_id')[column].rolling(window=w, min_periods=1).std().fillna(0).reset_index(level=0, drop=True)

In [148]:
train_df.shape, test_df.shape

((20631, 61), (13096, 60))

In [ ]:
# Step 4 - clip RUL at 125 cycles in train_df only. Engines with many flights left all look the same. In sensor terms, it is pointless to distinguish between a RUL of 300 and 400
# Focusing the model on the 0-125 range improves performance.
max_value = 125
train_df['RUL'] = train_df['RUL'].clip(upper=max_value)

In [170]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Index(['engine_id', 'cycle', 'op1', 'op2', 'op3', 'sensor1', 'sensor2',
#        'sensor3', 'sensor4', 'sensor5', 'sensor6', 'sensor7', 'sensor8',
#        'sensor9', 'sensor10', 'sensor11', 'sensor12', 'sensor13', 'sensor14',
#        'sensor15', 'sensor16', 'sensor17', 'sensor18', 'sensor19', 'sensor20',
#        'sensor21'],
#       dtype='object')

In [177]:
# Step 5 - Normalize features in both train_df and test_df
# Scale every feature column to the 0-1 range using MinMaxScaler. 
# Fit the scaler on training data only, then use that same fitted scaler to transform the test data.
# Save the scaler to disk -- you'll need it later in your API
normalizer = MinMaxScaler()

feature_cols = [col for col in train_df.columns if col not in ['engine_id', 'cycle', 'RUL']]

print(len(feature_cols))

58


In [178]:
train_df[feature_cols] = normalizer.fit_transform(train_df[feature_cols])
test_df[feature_cols] = normalizer.transform(test_df[feature_cols])

In [179]:
import joblib

In [182]:
# Step 6 - Save cleaned and feature engineered train and data to CSVs and the normalizer scaler to a pkl all on disk

In [183]:
joblib.dump(train_df, '../data/features/train_features.csv')
joblib.dump(test_df, '../data/features/test_features.csv')

['../data/features/test_features.csv']

In [184]:
joblib.dump(normalizer, '../models/scaler.pkl')

['../models/scaler.pkl']